###Get storage account parameter
This parameter is passed in from the ADF pipeline and varies by the environment in which the pipeline executes, e.g. `edapadlsprocurement` for DEV against `edapadlsqaprocurement` for QA.

In [0]:
dbutils.widgets.text('storage_account', '', '')
dbutils.widgets.text('key_vault_scope', '', '')

key_vault_scope = dbutils.widgets.get('key_vault_scope') # pylint: disable=unused-variable

### Run shared notebook

In [0]:
%run /dxcore/Utilities/MailAlerts $key_vault_scope=key_vault_scope

In [0]:
storage_account = dbutils.widgets.get('storage_account') # pylint: disable=unused-variable
print(f"Storage account: {storage_account}")

In [0]:
%run /dxcore/Utilities/Utilities $storage_account=storage_account

In [0]:
#%run /dxcore/Utilities/great-expectations-init $storage_account=storage_account $key_vault_scope=key_vault_scope

###Define input parameters and get configuration

In [0]:
dbutils.widgets.text('landing_partition', '', '')
dbutils.widgets.text('params', '', '')
dbutils.widgets.text('ingestion_ts', '', '')

landing_partition   = dbutils.widgets.get('landing_partition')
json_params         = dbutils.widgets.get('params')
ingestion_ts        = dbutils.widgets.get('ingestion_ts')

In [0]:
if not ingestion_ts:
  ingestion_ts = str(datetime.datetime.now())
#
# Parse JSON from input parameter
#
parsed_params = json.loads(json_params)
table_name = parsed_params['DataMovementShortName']

try:
  dataset = c.get_dataset_by_table_name(table_name)
  print(dataset)
  
except Exception as e:
  dbutils.notebook.exit(get_error_payload(
      f"Error in parsing json, Error: {str(e)}", dataset.name))
#
# Get ingestion timestamp from a parameter passed into this notebook from ADF. If, for some reason, it has not been
# passed in successfully, set it to the current time.


# if table_name.__contains__('cpp'):
#   landing_directory = '/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])+'/*.gz'
# else:
if parsed_params['SinkDatasetType'] == 'adls':
  landing_directory ='/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])+'/*.csv'
else:
  landing_directory   ='/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])+'/*.'+parsed_params['SinkDatasetType']
# landing_directory   = parsed_params['BronzeSource']
 
# bronze_directory    =  parsed_params['BronzeDest']
bronze_directory = '/'.join(parsed_params['SinkLandingDirectory'].split('/')[1:])
data_source_name = bronze_directory.split('/')[0] # pylint: disable=unused-variable
landing_file_format = parsed_params['SinkDatasetType']
print(landing_directory)
print(bronze_directory)

In [0]:
try:
  dataset = c.get_dataset_by_table_name(table_name)
  if not dataset:
    raise Exception(f"Unable to find configuration for dataset '{table_name}'")
except Exception as e:
  dbutils.notebook.exit(get_error_payload(
      f"Unable to find configuration for dataset, Error: {str(e)}", dataset.name))

if not dataset:
  print(f'Table Name {table_name} is not found in the configuration file')
  dbutils.notebook.exit(get_error_payload(f'Dataset {table_name} is not found in the configuration file', ''))

###Get Landing zone config data

In [0]:
landing_conf         = dataset.zones['landing']
landing_folder       = landing_conf.folder
landing_partitionBy  = landing_conf.databricks.partition_by
landing_path         = dataset.dataset_uri(landing_folder, landing_directory)

print(f"Landing path:\t\t\t{landing_path}")
print(f"Landing partition column:\t{landing_partitionBy}")
print(f"Landing partition folder:\t{landing_folder}")

###Get Bronze zone config data

In [0]:
bronze_conf          = dataset.zones['bronze']
# bronze_schema_main   = bronze_conf.databricks.schema
bronze_format        = bronze_conf.databricks.format
bronze_write_mode    = dataset.mode #bronze_conf.databricks.mode 
bronze_write_options = bronze_conf.databricks.options
# bronze_write_options = dataset.mergeSchema #bronze_conf.databricks.options
bronze_partition_col = bronze_conf.databricks.partition_by
# bronze_directory     = f"{bronze_conf.folder}/{'/'.join(landing_directory.split('/')[1:])}"
bronze_path          = dataset.dataset_uri(zone='bronze',dataset_name=bronze_directory)

print(f"Bronze format:\t\t\t{bronze_format}")
print(f"Bronze write mode:\t\t{bronze_write_mode}")
print(f"Bronze write options:\t\t{bronze_write_options}")
print(f"Bronze partition column:\t{bronze_partition_col}")
print(f"Bronze path:\t\t\t{bronze_path}")

###Set legacy date support for SQL

In [0]:
spark.conf.set('spark.sql.legacy.parquet.datetimeRebaseModeInRead', 'LEGACY')
spark.conf.set('spark.sql.legacy.parquet.datetimeRebaseModeInWrite', 'LEGACY')

###Set the number of partitions Spark uses for shuffling to match the number of cores in the cluster

In [0]:
spark.conf.set('spark.sql.shuffle.partitions', sc.defaultParallelism)

### Enable caching and adaptive execution

In [0]:
spark.conf.set('spark.databricks.io.cache.enabled', 'true')
spark.conf.set('spark.sql.adaptive.enabled', 'true')

###Read dataframe from Landing container

In [0]:
try:  # pylint: disable=R1702
  
  if landing_file_format in ['gz']:
      if table_name.__contains__("dg_product") or table_name.__contains__("wf_product") or table_name.__contains__("dg_store") or table_name.__contains__("wf_store"):
          infer_schema = 'false'
      else:
          infer_schema = 'true'
      landing_df = read_gz_files(landing_path, infer_schema)
      
      for name in landing_df.schema.names:
        landing_df = landing_df.withColumnRenamed(name,name.replace(' ', '_'))

  if landing_file_format in ['csv','adls']:
    if table_name.__contains__("pdi"):
      if table_name.__contains__("pr"):
        landing_df = read_pdi_files(landing_path, '\"')
      else:
        landing_df = read_pdi_files(landing_path, '\\')
    elif table_name.__contains__("syndigo"):
      landing_df = read_csv_files(landing_path)
      landing_df = landing_df.select([col(col).alias(re.sub("[^0-9a-zA-Z$]+","",col)) for col in landing_df.columns])
    else:
      landing_df = read_csv_files(landing_path)
    for name in landing_df.schema.names:
      landing_df = landing_df.withColumnRenamed(name,name.replace(' ', '_'))
      
    if table_name.__contains__("home_panel_summary_weekly_backfill"):
      landing_df= landing_df.where(landing_df.filepath != '2021/11/02/02/2021/07/05/home_panel_summary.csv')
      landing_df= landing_df.where(landing_df.filepath != '2021/11/02/02/2021/07/12/home_panel_summary.csv')

    elif table_name.__contains__("normalization_stats_weekly_backfill"):
      landing_df= landing_df.where(landing_df.filepath != '2021/11/02/02/2021/07/05/normalization_stats.csv')
      landing_df= landing_df.where(landing_df.filepath != '2021/11/02/02/2021/07/12/normalization_stats.csv')

    elif table_name.__contains__("core_poi-patterns_weekly_backfill"):
      list_cpp = ['2021/07/22/18/2021/07/05/core_poi-patterns-part1.csv'
                 ,'2021/07/22/18/2021/07/05/core_poi-patterns-part2.csv'
                 ,'2021/07/22/18/2021/07/05/core_poi-patterns-part3.csv'
                 ,'2021/07/22/18/2021/07/05/core_poi-patterns-part4.csv'
                 ,'2021/07/22/18/2021/07/05/core_poi-patterns-part5.csv'
                 ,'2021/07/22/18/2021/07/05/core_poi-patterns-part6.csv'
                 ,'2021/07/22/18/2021/07/12/core_poi-patterns-part1.csv'
                 ,'2021/07/22/18/2021/07/12/core_poi-patterns-part2.csv'
                 ,'2021/07/22/18/2021/07/12/core_poi-patterns-part3.csv'
                 ,'2021/07/22/18/2021/07/12/core_poi-patterns-part4.csv'
                 ,'2021/07/22/18/2021/07/12/core_poi-patterns-part5.csv'
                 ,'2021/07/22/18/2021/07/12/core_poi-patterns-part6.csv']
      for files in list_cpp:
        landing_df= landing_df.where(landing_df.filepath != files )

  elif landing_file_format == 'parquet':
    landing_df = get_dataset_parquet(landing_path) #spark.read.format('parquet').load(landing_path)
  elif landing_file_format == 'orc':
    landing_df = get_dataset_orc(landing_path)
  elif landing_file_format == 'text':
    landing_df = get_dataset_delimited(landing_path,parsed_params['SinkColumnDelim'],parsed_params['SinkQuoteChar']
        ,parsed_params['SinkEscapeChar']
        ,'True'
      )
      #.option("multiline", "true").option("quote", '"').option("escape", "\\").option("escape", '"')
  else:
    pass

    #
    # Get values of partition columns
    #
  ingestion_dt = landing_partition.split('/')[0].split('=')[1]
  job_id       = landing_partition.split('/')[1].split('=')[1]
    #
    # Filter by partition columns
    # Make sure that created_dtm is indeed a timestamp
    #

  if 'created_dtm' not in landing_df.columns:
    landing_df = landing_df.withColumn('created_dtm', F.lit(ingestion_ts).cast('timestamp'))
  if 'last_updated_dtm' not in landing_df.columns:
    landing_df = landing_df.withColumn('last_updated_dtm', F.lit(ingestion_ts).cast('timestamp'))
    #
    # Change datatypes, if necessary
    #
  landing_df = (landing_df
                  .withColumn('created_dtm',      F.col('created_dtm').cast('timestamp'))
                  .withColumn('last_updated_dtm', F.col('created_dtm').cast('timestamp'))
                  .withColumn('job_id',          lit(job_id))
                 )
except Exception as e:
  dbutils.notebook.exit(get_error_payload(
      f"Error in reading the dataframe from {landing_path}, Error: {str(e)}", dataset.name))

###Adding file name to text sources

In [0]:
try:
  if landing_file_format == 'text':
    file_name  = parsed_params['SinkFileName']
    landing_df = landing_df.withColumn("src_file_nm",  F.lit(file_name))
    landing_df.display()
except Exception as e:
  dbutils.notebook.exit(get_error_payload(
      f"Error in adding file name to text sources, Error: {str(e)}", dataset.name))

###Cast columns in landing DataFrame to match target bronze table

In [0]:
def normalize_column_name(string: str) -> str:  # pylint: disable=R1710
  if string:
    string = re.sub('[^0-9a-z]+', '_', string.lower())
    if string and string[0] == '_':
      string = string[1:]
    if string and string[-1] == '_':
      string = string[:-1]
    return string

try:  # pylint: disable=R1702
  landing_schema  = get_schema_from_df(landing_df)
  try:
    bronze_schema = get_schema_from_df(spark.read.format(bronze_format).load(bronze_path))
  except Exception as e:
    bronze_schema = []
    
  if bronze_schema:
    for landing_column, landing_datatype in landing_schema.items():
      bronze_datatype = bronze_schema.get(landing_column)
      if not bronze_datatype:
        #
        # Check whether column is in Bronze table in normalized form
        #
        normalized_landing_column = normalize_column_name(landing_column)
        bronze_datatype = bronze_schema.get(normalized_landing_column)
        if bronze_datatype:
          print(f"Renaming column {landing_column} to {normalized_landing_column}")
          landing_df = landing_df.withColumnRenamed(landing_column, normalized_landing_column)
          if bronze_datatype != landing_datatype:
            print(f"For column {landing_column}: Landing: {landing_datatype} <> Bronze: {bronze_datatype}")
            landing_df = landing_df.withColumn(normalized_landing_column, F.col(normalized_landing_column).cast(bronze_datatype))            
        else:
          print(f"Dropping {landing_column} from the landing DataFrame")
          landing_df = landing_df.drop(landing_column)
          #if landing_column in translation_columns:
          #  print(f"Landing column {landing_column} is not in bronze, but is a translatable column")
          #elif landing_column == dataset.uuid.lower():
          #  print(f"Landing column {landing_column} is in bronze, but is the uuid used for translation")
          #else:
          #  print(f"Dropping {landing_column} from the landing DataFrame")
          #  landing_df = landing_df.drop(landing_column)
      elif bronze_datatype != landing_datatype:
        print(f"For column {landing_column}: Landing: {landing_datatype} <> Bronze: {bronze_datatype}")
        landing_df = landing_df.withColumn(landing_column, F.col(landing_column).cast(bronze_datatype))
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Error casting landing columns to bronze for {table_name} dataset, Error: {str(e)}", dataset.name))
  

###Get row count of landing DataFrame

In [0]:
try:
  row_count = landing_df.count()
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Error getting row count for the landing {table_name} datasets, Error: {str(e)}", dataset.name))
  
  
print(f'Row count = {row_count:,}')

###Hash the PKs into a new column for easier joining downstream###
It is easier and more efficient to join with a single column than with multiple columns. In order to facilitate joining with a single column in the Silver zone, we can concatenate and hash the PKs. We first check the configuration to see whether this dataset is configured with a hash column. If it is, we use PySpark's `concat_ws` function to concatentate the primary keys with a `-` separator and then apply the `md5` function to has the result. If the PK is not compound, i.e. a single column, we can hash it directly without using `concat_ws` after casting to `string`.

In [0]:
try:
  if dataset.hash_column:
    print(f"New column `{dataset.hash_column}` will be the MD5 hash of PKs: {dataset.primary_key}")
    hash_column = get_hash_column(dataset.primary_key)
    landing_df = landing_df.withColumn(dataset.hash_column, hash_column)
except Exception as e:
  dbutils.notebook.exit(get_error_payload(
      f"Error in hashing the PKs, Error: {str(e)}", dataset.name))

### Run Great Expectation tests for landing to bronze

In [0]:
# try:
#   landing_df_ge = landing_df.limit(100)
#   result = get_ge_result(table_name,data_source_name,landing_df_ge)
#   if result == 0:
#     print("No expectations created")
#   else:
#     assert result.success
# except Exception as e:
#   dbutils.notebook.exit(get_error_payload(
#       f"Error in running Great Expectations tests for landing to bronze, Error: {str(e)}", dataset.name))

###Save landing DataFrame to Bronze zone

In [0]:
#sc.setJobDescription(f'Write {table_name} dataset to bronze container')

if not bronze_write_options:
  bronze_write_options = {}
  
try:
  if bronze_partition_col:
    (landing_df
     .withColumn(bronze_partition_col, F.lit(ingestion_dt).cast('date'))
     .write
     .format(bronze_format)
#      .options(**bronze_write_options)
#      .option("mergeSchema", bronze_write_options)
     .mode(bronze_write_mode)
     .partitionBy(bronze_partition_col)
     .save(bronze_path)
    )
  else:
    (landing_df
     .withColumn(bronze_partition_col, F.lit(ingestion_dt).cast('date'))
     .write
     .format(bronze_format)
#      .options(**bronze_write_options)
#      .option("mergeSchema", bronze_write_options)
     .mode(bronze_write_mode)
     .save(bronze_path)
    )    
except Exception as e:
  
  dbutils.notebook.exit(get_error_payload(
    f"Error writing {dataset.system}/{dataset.name} dataset to bronze container, Error: {str(e)}", dataset.name))
  

###Returns success message to caller (in this case, Azure Data Factory)

In [0]:
dbutils.notebook.exit(json.dumps({
  "status": PipelineStatus.SUCCESS.value
  ,"dataset": dataset.name
  ,"row_count": row_count
  ,"partition_count": 0
  ,"is_translatable": dataset.is_translatable
  ,"error_msg": None
}))